In [2]:
import os
import requests
from tqdm import tqdm
import gzip
import shutil
from tqdm import tqdm
import pandas as pd

In [ ]:
# Create the GHCN dataset
years = range(2021, 2024) # Array of years to download
base_url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/" # Retrieve GHCN data by year https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/

def download_ghcn_data(years, base_url, output_dir="ghcn_data"):
    os.makedirs(output_dir, exist_ok=True)
    
    for year in tqdm(years):
        file_name = f"{year}.csv.gz"
        url = f"{base_url}{file_name}"
        output_path = os.path.join(output_dir, file_name)
        
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(output_path, 'wb') as f:
                shutil.copyfileobj(response.raw, f)
        else:
            print(f"Failed to download {file_name}: Status code {response.status_code}")

download_ghcn_data(years, base_url)

100%|██████████| 3/3 [00:11<00:00,  3.88s/it]


In [4]:
# Load downloaded files into dataframes
year_dfs = []
for year in tqdm(years):
    file_path = os.path.join("ghcn_data", f"{year}.csv.gz")
    with gzip.open(file_path, 'rt') as f:
        df = pd.read_csv(f, header=None, index_col=False, names=['ID', 'Date', 'Element', 'Value', 'MFlag', 'QFlag', 'SFlag', 'OBS-TIME'])
        year_dfs.append(df)

# Combine all years into a single dataframe
combined_df = pd.concat(year_dfs)

100%|██████████| 3/3 [01:07<00:00, 22.39s/it]


In [5]:
# Filter for specific elements (e.g., TMAX, TMIN, PRCP)
filtered_df = combined_df[combined_df['Element'].isin(['TMAX', 'TMIN'])]
# filtered_df.to_csv(f"ghcn_temp_{years[0]}_{years[-1]}_filtered.csv", index=False)


In [ ]:
from noaa_ghcn import GHCN

# Get station metadata
ghcn = GHCN()
ghcn.stations

In [ ]:
# Merge all observations with station metadata
merged_df = pd.merge(combined_df, ghcn.stations, left_on='ID', right_on='ID', how='left')
merged_df.to_csv(f"ghcn_{years[0]}_{years[-1]}_with_stations.csv", index=False)

# This works, but is extremely slow for large datasets
# indx = ghcn.filter_inventory(station_ids=ghcn_stations_subset, start_date= datetime.datetime(2020, 1, 1), end_date=datetime.datetime(2022, 12, 31), elements=["TMAX", "TMIN", "PRCP"])
# df = ghcn.load_data(indx)
# df.to_csv("ghcn_data_2020_2022.csv", index=False)

In [ ]:
# Merge filtered data with station metadata
merged_df = pd.merge(filtered_df, ghcn.stations, left_on='ID', right_on='ID', how='left')
merged_df.to_csv(f"ghcn_{years[0]}_{years[-1]}_temperature.csv", index=False)

In [ ]:
import pandas as pd
import requests

def download_station_metadata():
    """Download GHCN-Daily station metadata."""
    print("Downloading station metadata...")
    url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
    response = requests.get(url)
    
    # Parse fixed-width format
    # Format: ID LATITUDE LONGITUDE ELEVATION STATE NAME
    stations = []
    for line in response.text.split('\n'):
        if len(line) < 41:
            continue
        station = {
            'station_id': line[0:11].strip(),
            'lat': float(line[12:20].strip()),
            'lon': float(line[21:30].strip()),
            'elevation': float(line[31:37].strip()) if line[31:37].strip() else None,
            'name_of_station': line[41:71].strip() if len(line) > 41 else ''
        }
        stations.append(station)
    
    return pd.DataFrame(stations)

download_station_metadata()

,station_id,lat,lon,elevation,name_of_station
0,ACW00011604,17.1167,-61.7833,10.1,ST JOHNS COOLIDGE FLD
1,ACW00011647,17.1333,-61.7833,19.2,ST JOHNS
2,AE000041196,25.3330,55.5170,34.0,SHARJAH INTER. AIRP
3,AEM00041194,25.2550,55.3640,10.4,DUBAI INTL
4,AEM00041217,24.4330,54.6510,26.8,ABU DHABI INTL
...,...,...,...,...,...
129652,ZI000067969,-21.0500,29.3670,861.0,WEST NICHOLSON
129653,ZI000067975,-20.0670,30.8670,1095.0,MASVINGO
129654,ZI000067977,-21.0170,31.5830,430.0,BUFFALO RANGE
129655,ZI000067983,-20.2000,32.6160,1132.0,CHIPINGE


In [8]:
import gzip 
import datetime

def parse_dly_format(content, station_id):
    """Parse GHCN-Daily .dly fixed-width format."""
    records = []
    
    for line in content.split('\n'):
        if len(line) < 21:
            continue
            
        station = line[0:11]
        year = int(line[11:15])
        month = int(line[15:17])
        element = line[17:21]  # e.g., TMAX, TMIN, PRCP
        
        # Only process temperature elements
        if element not in ['TMAX', 'TMIN', 'TAVG']:
            continue
        
        # Parse daily values (positions 21+ contain 31 days of data)
        for day in range(1, 32):
            pos = 21 + (day - 1) * 8
            if pos + 5 > len(line):
                break
                
            value_str = line[pos:pos+5].strip()
            if not value_str or value_str == '-9999':
                continue
                
            value = int(value_str) / 10.0  # Values are in tenths of degrees C
            
            try:
                date = datetime.datetime(year, month, day)
                records.append({
                    'ID': station,
                    'DATE': date.strftime('%Y-%m-%d'),
                    'ELEMENT': element,
                    'DATA_VALUE': value
                })
            except ValueError:
                # Invalid date (e.g., Feb 30)
                continue
    
    return pd.DataFrame(records)

def download_station_data(station_id, year=None):
    """
    Download daily data for a specific station.
    If year is None, downloads all available data (can be large!).
    """
    print(f"Downloading data for station {station_id}...")
    
    if year:
        # Download specific year
        url = f"https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/{year}.csv.gz"
        print(f"Downloading year {year}...")
    else:
        # Download all data for specific station (individual station file)
        url = f"https://www.ncei.noaa.gov/pub/data/ghcn/daily/all/{station_id}.dly"
        print(f"Downloading all data for {station_id}...")
    
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Failed to download: {url}")
        return None
    
    # Handle different formats
    if url.endswith('.csv.gz'):
        # Decompress and read CSV
        content = gzip.decompress(response.content).decode('utf-8')
        df = pd.read_csv(StringIO(content))
        # Filter for specific station if provided
        if station_id:
            df = df[df['ID'] == station_id]
        return df
    else:
        # Parse .dly format (fixed-width format)
        return parse_dly_format(response.text, station_id)
    
download_station_data("ACW00011647")

,ID,DATE,ELEMENT,DATA_VALUE
0,ACW00011647,1961-10-01,TMAX,27.2
1,ACW00011647,1961-10-01,TMIN,23.9
2,ACW00011647,2025-01-01,TMAX,29.0
3,ACW00011647,2025-01-02,TMAX,28.0
4,ACW00011647,2025-01-03,TMAX,28.0
...,...,...,...,...
1149,ACW00011647,2026-01-21,TAVG,25.5
1150,ACW00011647,2026-01-22,TAVG,24.3
1151,ACW00011647,2026-01-23,TAVG,25.2
1152,ACW00011647,2026-01-24,TAVG,25.8
